# Transformer 网络演示 (带 Embedding 模型)
本脚本演示了如何使用 PyTorch 搭建一个简单的 Sequence-to-Sequence (Seq2Seq) Transformer 模型。
**本次更新新增了独立的 Embedding 模块和可视化功能。**

**核心特性：**
- **Embedding 模型**：显式封装了 Token Embedding 和 Position Embedding。
- **Embedding 可视化**：训练结束后，展示所学到的数字向量分布。
- **步进可视化**：推理过程打印每一步的决策。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import random
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
# ==========================================
# 1. 配置与超参数 (Configuration)
# ==========================================
class Config:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 数据参数
    vocab_size = 20  # 词汇表大小 (整数 0..19)
    seq_len = 10     # 输入序列长度
    batch_size = 64
    num_samples = 2000 
    
    # 模型参数
    d_model = 128    # 嵌入维度 (Embedding dimension)
    nhead = 4        
    num_encoder_layers = 2
    num_decoder_layers = 2
    dim_feedforward = 512
    dropout = 0.1
    
    # 训练参数
    learning_rate = 0.0005
    epochs = 50      

print(f"当前使用的设备: {Config.device}")

In [ ]:
# ==========================================
# 2. 数据准备
# ==========================================
def generate_reverse_data(num_samples, seq_len, vocab_size):
    # 0:PAD, 1:BOS, 2:EOS. 实际数据从 3 开始
    src_data = []
    tgt_data = []
    
    for _ in range(num_samples):
        seq = [random.randint(3, vocab_size-1) for _ in range(seq_len)]
        src = seq
        tgt = [1] + seq[::-1] + [2]
        src_data.append(src)
        tgt_data.append(tgt)
        
    return torch.LongTensor(src_data), torch.LongTensor(tgt_data)

src_data, tgt_data = generate_reverse_data(Config.num_samples, Config.seq_len, Config.vocab_size)

from torch.utils.data import TensorDataset, DataLoader
dataset = TensorDataset(src_data, tgt_data)
train_loader = DataLoader(dataset, batch_size=Config.batch_size, shuffle=True)

In [ ]:
# ==========================================
# 3. 模型定义 - 新增 TransformerEmbedding
# ==========================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x: [batch, seq_len, d_model]
        return x + self.pe[:, :x.size(1), :]

class TransformerEmbedding(nn.Module):
    """
    Embedding 模型：将 Token ID 转换为向量，并加上位置编码。
    """
    def __init__(self, vocab_size, d_model, max_len=5000, dropout=0.1):
        super(TransformerEmbedding, self).__init__()
        # 1. Token Embedding: 学习每个词的向量表示
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        # 2. Positional Encoding: 注入位置信息
        self.pos_emb = PositionalEncoding(d_model, max_len)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        tok_emb = self.tok_emb(x)
        pos_emb = self.pos_emb(tok_emb * math.sqrt(tok_emb.size(-1))) # 缩放通常在这里或PosEnc里做，这里遵循标准实现
        return self.dropout(pos_emb)

class TransformerSeq2Seq(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_encoder_layers, num_decoder_layers, dim_feedforward, dropout=0.1):
        super(TransformerSeq2Seq, self).__init__()
        
        # 使用独立的 Embedding 模块
        self.embedding_layer = TransformerEmbedding(vocab_size, d_model, dropout=dropout)
        
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        
        self.fc_out = nn.Linear(d_model, vocab_size)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, src_key_padding_mask=None, tgt_key_padding_mask=None):
        # 通过 Embedding 模块
        src_emb = self.embedding_layer(src)
        tgt_emb = self.embedding_layer(tgt)
        
        out = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )
        
        return self.fc_out(out)

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

model = TransformerSeq2Seq(
    Config.vocab_size, Config.d_model, Config.nhead, 
    Config.num_encoder_layers, Config.num_decoder_layers, 
    Config.dim_feedforward, Config.dropout
).to(Config.device)

In [ ]:
# ==========================================
# 4. 训练循环
# ==========================================
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=Config.learning_rate)

print("开始训练...")
losses = []
for epoch in range(Config.epochs):
    model.train()
    total_loss = 0
    
    for src, tgt in train_loader:
        src, tgt = src.to(Config.device), tgt.to(Config.device)
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]
        tgt_mask = model.generate_square_subsequent_mask(tgt_input.size(1)).to(Config.device)
        
        optimizer.zero_grad()
        output = model(src, tgt_input, tgt_mask=tgt_mask)
        loss = criterion(output.reshape(-1, Config.vocab_size), tgt_output.reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"轮次 [Epoch {epoch+1}/{Config.epochs}] | 损失: {avg_loss:.4f}")

print("训练完成。")

In [ ]:
# ==========================================
# 5. 步进式推断
# ==========================================
def predict_step_by_step(model, src_seq, max_len=20):
    model.eval()
    src = torch.LongTensor([src_seq]).to(Config.device)
    tgt_tokens = [1]
    
    print(f"\n{'='*40}")
    print(f"输入序列: {src_seq}")
    
    for i in range(max_len):
        tgt = torch.LongTensor([tgt_tokens]).to(Config.device)
        tgt_mask = model.generate_square_subsequent_mask(tgt.size(1)).to(Config.device)
        
        with torch.no_grad():
            output = model(src, tgt, tgt_mask=tgt_mask)
        
        last_token_logits = output[:, -1, :]
        probabilities = torch.softmax(last_token_logits, dim=-1)
        top_probs, top_indices = torch.topk(probabilities, 3, dim=-1)
        
        next_token = top_indices[0, 0].item()
        
        print(f"Step {i+1}: 上下文 {tgt_tokens} -> 预测: {next_token}")
        
        if next_token == 2:
            tgt_tokens.append(next_token)
            break
            
        tgt_tokens.append(next_token)
    return tgt_tokens[1:-1]

test_seq = [5, 10, 15, 7]
predict_step_by_step(model, test_seq)

In [ ]:
# ==========================================
# 6. Embedding 可视化 (新增)
# ==========================================
def visualize_embeddings(model):
    """
    使用 PCA 将 128维 的 Embedding 降维到 2维 并画图
    """
    model.eval()
    # 获取 Embedding 层的权重矩阵 (vocab_size, d_model)
    # 注意: 我们提取的是 Embedding 类中的 tok_emb
    embeddings = model.embedding_layer.tok_emb.weight.detach().cpu().numpy()
    
    # 使用 PCA 降维
    pca = PCA(n_components=2)
    reduced_embeddings = pca.fit_transform(embeddings)
    
    # 绘图
    plt.figure(figsize=(10, 8))
    plt.scatter(reduced_embeddings[:, 0], reduced_embeddings[:, 1], c='blue', alpha=0.6)
    
    # 给每个点加上数字标签
    for i, (x, y) in enumerate(reduced_embeddings):
        label = f"{i}"
        if i == 0: label += " (PAD)"
        elif i == 1: label += " (BOS)"
        elif i == 2: label += " (EOS)"
        
        plt.annotate(label, (x, y), xytext=(5, 2), textcoords='offset points')
        
    plt.title("Token Embeddings 可视化 (PCA)")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

print("正在生成 Embedding 可视化图...")
visualize_embeddings(model)